<div style='background:#0b0e10;color:#f4f5f6;border:1px solid #2c3136;border-radius:18px;padding:28px 22px;text-align:center'><div style='font-family:monospace;font-size:12px;letter-spacing:.22em;color:#9aa1aa'>MINEIRO · PYTHON · v2.0.0a1</div><h1 style='font-size:36px;margin:10px 0 4px'>MINEIRO USERNAME INTELLIGENCE</h1><p style='font-size:17px;color:#a3a9b0;margin:0'>OSINT Investigation Workbench</p><p style='font-size:13px;color:#7f8790;margin-top:12px'>coleta → evidência → correlação → avaliação</p></div>

**Uso:** *Ambiente de execução → Executar tudo*. A interface aparece **aqui mesmo, no notebook**: sem servidor, sem proxy e sem build.

> Use apenas identificadores públicos que você esteja autorizado a investigar. Um `FOUND` é uma observação, não prova de identidade.

## 1 · Instalar
Instala o pacote `mineiro` (≈80 kB, depende só de `httpx`). Troque `BRANCH` se estiver testando uma branch.

In [ ]:
BRANCH = 'main'  # antes do merge do PR do Python, use 'feat/mineiro-py'
%pip install -q "mineiro @ git+https://github.com/Ridd1kulusC0d3r/mineirohml.git@$BRANCH#subdirectory=mineiro-py"

import mineiro
from mineiro import load_registry
registry = load_registry()
print(f'Mineiro {mineiro.__version__} · {len(registry)} detectores ({registry.stats()["scannableDetectors"]} escaneáveis)')

## 2 · Interface
Escolha o alvo, o preset (**quick** 19 sites · **standard** 49 com Evidence Engine e baseline · **full** 984 em duas fases), o idioma e o requisito de inteligência.

In [ ]:
from mineiro.ui import launch
launch()

## 3 · Pela API Python (opcional)
O mesmo motor, sem widgets. No Colab dá para usar `await` direto numa célula.

In [ ]:
from mineiro import Scanner, build_assessment
from mineiro.export import write_report

async with Scanner(preset='quick') as scanner:
    results = await scanner.scan('forgejo')

assessment = build_assessment(results, 'forgejo', lang='pt')
for e in assessment['evidence']:
    if e['status'] == 'found':
        print(e['platform_name'], e['source_quality'], e['intelligence_priority_score'], e['url'])

path, manifest = write_report(assessment, results, 'mineiro-reports', 'html')
print('relatório:', path)

## 4 · Grafo, casos e diff (opcional)
Consulta local somente leitura (`MATCH`, `EDGE`, `SHARED`, `PATH`), export Cypher e Diff Intelligence entre duas coletas.

In [ ]:
from mineiro.graph import run_graph_query, graph_to_cypher, add_variant_candidates
from mineiro.variants import generate_username_variants
from mineiro.cases import CaseStore, diff_collections

graph = add_variant_candidates(assessment['graph'], 'forgejo', generate_username_variants('forgejo'))
print(run_graph_query('PATH from=TARGET to=PROFILE', graph)['summary'])
print(run_graph_query('MATCH candidate=true AND similarity>=90', graph)['summary'])

store = CaseStore('mineiro-reports/cases.sqlite')
case_id = store.get_or_create_case('forgejo')
store.add_snapshot(case_id, results, 'quick')
print(store.list_cases())

## 5 · Verificar integridade
Cada exportação gera um `.manifest.json` com o SHA-256 do arquivo. Isso prova integridade do conteúdo, não autoria nem cadeia de custódia.

In [ ]:
from mineiro.export import verify_manifest
print('íntegro:', verify_manifest(path))